# 01 - Exploration des sources

Diagnostic de chaque source nettoyée par `src/load.py` (parquets dans `data/processed/`) :
dimensions, types, valeurs manquantes, exemples, doublons sur la clé, couverture
communale et valeurs aberrantes. **Aucune jointure entre sources** à ce stade, sauf
dans la dernière section, qui construit seulement le squelette des clés du panel.

Prérequis : `python scripts/download_data.py` puis `python src/load.py`.
Les parquets manquants sont régénérés à la volée.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent  # notebook lancé depuis notebooks/
sys.path.insert(0, str(ROOT / "src"))
import load  # noqa: E402

PROCESSED = ROOT / "data" / "processed"
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)


def get(name: str, **kwargs) -> pd.DataFrame:
    path = PROCESSED / f"{name}.parquet"
    if not path.exists():
        load.write_processed(name)
    return pd.read_parquet(path, **kwargs)


def diagnostic(df: pd.DataFrame, key: list[str] | None = None, n_types: int = 40) -> None:
    """Shape, types, % de valeurs manquantes, 3 lignes d'exemple, doublons sur la clé."""
    print(f"shape : {df.shape}")
    info = pd.DataFrame({"type": df.dtypes.astype(str), "pct_manquant": (df.isna().mean() * 100).round(2)})
    display(info.head(n_types).T)
    display(df.sample(3, random_state=0) if len(df) > 3 else df)
    if key:
        dup = df.duplicated(key, keep=False)
        print(f"doublons sur {key} : {dup.sum()} lignes ({df[dup][key].drop_duplicates().shape[0]} clés)")

## 1. Municipales 2014, 2020, 2026

Tables longues : une ligne par commune (x secteur ou section) x liste ou candidat.
La clé « commune x scrutin x tour » se vérifie sur la table agrégée par commune
(`load.communes_from_muni`).

In [2]:
muni = {y: get(f"municipales_{y}") for y in (2014, 2020, 2026)}
muni_all = pd.concat(muni.values(), ignore_index=True)
communes = load.communes_from_muni(muni_all)
for y, d in muni.items():
    print(f"=== {y}")
    diagnostic(d, n_types=29)

=== 2014
shape : (382613, 27)


,annee,tour,code_insee,secteur,code_dep,libelle_dep,code_commune_source,libelle_commune,mode_scrutin,inscrits,abstentions,votants,blancs_et_nuls,exprimes,nuance,sexe,nom,prenom,liste,voix,pct_voix_ins,pct_voix_exp,elu,sieges_cm,sieges_secteur,sieges_cc,source
type,int64,int64,string,string,string,string,string,string,str,Int64,Int64,Int64,Int64,Int64,string,string,string,string,string,Int64,float64,float64,boolean,Int64,Int64,Int64,str
pct_manquant,0.0,0.0,0.0,99.21,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,76.65,0.0,0.0,0.0,93.08,0.0,0.0,0.0,0.0,93.09,93.09,93.09,0.0


,annee,tour,code_insee,secteur,code_dep,libelle_dep,code_commune_source,libelle_commune,mode_scrutin,inscrits,abstentions,votants,blancs_et_nuls,exprimes,nuance,sexe,nom,prenom,liste,voix,pct_voix_ins,pct_voix_exp,elu,sieges_cm,sieges_secteur,sieges_cc,source
50717,2014,2,33386,<NA>,33,GIRONDE,386,Saint-Cibard,plurinominal,172,45,127,7,120,NC,M,ROUSSILLE,Jean-Claude,<NA>,16,9.30,13.33,False,<NA>,<NA>,<NA>,muni2014_t2_moins1000
131772,2014,1,14548,<NA>,14,CALVADOS,548,Rucqueville,plurinominal,139,28,111,6,105,<NA>,F,CLAIR,Ginette,<NA>,60,43.16,57.14,True,<NA>,<NA>,<NA>,muni2014_t1_moins1000
51523,2014,2,34322,<NA>,34,HERAULT,322,Valflaunès,plurinominal,584,126,458,10,448,NC,F,BEL,Véronique,<NA>,202,34.59,45.09,True,<NA>,<NA>,<NA>,muni2014_t2_moins1000


=== 2020
shape : (418734, 29)


,annee,tour,code_insee,secteur,code_dep,libelle_dep,code_commune_source,libelle_commune,mode_scrutin,inscrits,abstentions,votants,blancs,nuls,exprimes,num_panneau,nuance,sexe,nom,prenom,liste,voix,pct_voix_ins,pct_voix_exp,elu,sieges_cm,sieges_secteur,sieges_cc,source
type,int64,int64,string,string,string,string,string,string,str,Int64,Int64,Int64,Int64,Int64,Int64,string,string,string,string,string,string,Int64,float64,float64,boolean,Int64,Int64,Int64,str
pct_manquant,0.0,0.0,0.0,99.76,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,94.09,0.0,0.0,0.0,0.0,94.09,94.09,94.09,0.0


,annee,tour,code_insee,secteur,code_dep,libelle_dep,code_commune_source,libelle_commune,mode_scrutin,inscrits,abstentions,votants,blancs,nuls,exprimes,num_panneau,nuance,sexe,nom,prenom,liste,voix,pct_voix_ins,pct_voix_exp,elu,sieges_cm,sieges_secteur,sieges_cc,source
355787,2020,1,81259,<NA>,81,Tarn,259,Saint-Julien-Gaulène,plurinominal,192,23,169,1,2,166,19,NC,M,OLIVIER,Richard,<NA>,54,28.13,32.53,False,<NA>,<NA>,<NA>,muni2020_t1_moins1000
45489,2020,1,05152,<NA>,05,Hautes-Alpes,152,Saint-Maurice-en-Valgodemard,plurinominal,138,42,96,4,0,92,3,NC,F,DUMAS,Julie,<NA>,74,53.62,80.43,True,<NA>,<NA>,<NA>,muni2020_t1_moins1000
141871,2020,1,29115,<NA>,29,Finistère,115,Lannédern,plurinominal,241,93,148,0,0,148,5,NC,M,COCHENNEC,Stéphane,<NA>,94,39.00,63.51,True,<NA>,<NA>,<NA>,muni2020_t1_moins1000


=== 2026
shape : (54119, 29)


,annee,tour,code_insee,secteur,code_dep,libelle_dep,code_commune_source,libelle_commune,mode_scrutin,inscrits,abstentions,votants,blancs,nuls,exprimes,num_panneau,nuance,sexe,nom,prenom,liste,liste_abrege,voix,pct_voix_ins,pct_voix_exp,elu,sieges_cm,sieges_cc,source
type,int64,int64,string,string,string,string,string,string,str,Int64,Int64,Int64,Int64,Int64,Int64,string,string,string,string,string,string,string,Int64,float64,float64,boolean,Int64,Int64,str
pct_manquant,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.03,78.38,92.22,92.22,92.22,0.0,0.0,0.0,0.0,0.0,0.0,0.0,57.6,0.0


,annee,tour,code_insee,secteur,code_dep,libelle_dep,code_commune_source,libelle_commune,mode_scrutin,inscrits,abstentions,votants,blancs,nuls,exprimes,num_panneau,nuance,sexe,nom,prenom,liste,liste_abrege,voix,pct_voix_ins,pct_voix_exp,elu,sieges_cm,sieges_cc,source
777,2026,1,02201,<NA>,02,Aisne,02201,Coeuvres-et-Valsery,liste,224,116,108,7,9,92,1,<NA>,<NA>,<NA>,<NA>,COEUVRES POUR TOUS,COEUVRES POUR TOUS,92,41.07,100.0,True,11,<NA>,muni2026_t1_communes
15131,2026,1,30174,<NA>,30,Gard,30174,Montaren-et-Saint-Médiers,liste,1089,307,782,7,6,769,2,<NA>,<NA>,<NA>,<NA>,POURSUIVONS ENSEMBLE VERS DEMAIN,POURSUIVONS ENSEMBLE VERS DEMAIN,426,39.12,55.4,True,12,2,muni2026_t1_communes
34063,2026,1,63188,<NA>,63,Puy-de-Dôme,63188,Laps,liste,471,186,285,17,25,243,1,<NA>,<NA>,<NA>,<NA>,VIVRE ENSEMBLE,VIVRE ENSEMBLE,243,51.59,100.0,True,15,<NA>,muni2026_t1_communes


In [3]:
# Doublons sur la clé commune x scrutin x tour : ils proviennent des secteurs
# (Paris, Lyon, Marseille : « SR ») et des sections électorales (Polynésie : « SN »).
key = ["code_insee", "annee", "tour"]
dup = communes[communes.duplicated(key, keep=False)]
print(f"{len(dup)} lignes commune x tour en double, {dup[key].drop_duplicates().shape[0]} clés")
display(dup.groupby(["annee", dup["secteur"].str[:2]]).code_insee.nunique().rename("n_communes").to_frame())

291 lignes commune x tour en double, 63 clés


n_communes
annee secteur            
2014  SN               28
      SR                3
2020  SN                8
      SR                3

In [4]:
# Nombre de communes par scrutin, tour et mode de scrutin
display(communes.groupby(["annee", "tour", "mode_scrutin"]).code_insee.nunique().unstack(["tour"]))

tour                    1     2
annee mode_scrutin             
2014  liste          9814  1750
      plurinominal  26719  4761
2020  liste          9958  1399
      plurinominal  24959  3180
2026  liste         34836  1526

In [5]:
def aberrations_muni(d: pd.DataFrame) -> pd.Series:
    c = d.drop_duplicates(["annee", "tour", "code_insee", "secteur"])
    checks = {
        "voix > inscrits": d.voix > d.inscrits,
        "voix > exprimes": d.voix > d.exprimes,
        "voix < 0": d.voix < 0,
        "pct_voix_exp hors [0, 100]": ~d.pct_voix_exp.dropna().between(0, 100),
    }
    out = {k: int(v.sum()) for k, v in checks.items()}
    part = c.votants / c.inscrits
    out.update({
        "votants > inscrits (communes)": int((c.votants > c.inscrits).sum()),
        "exprimes > votants (communes)": int((c.exprimes > c.votants).sum()),
        "participation hors [0, 1]": int((~part.dropna().between(0, 1)).sum()),
        "inscrits = 0": int((c.inscrits == 0).sum()),
        "inscrits manquants": int(c.inscrits.isna().sum()),
    })
    # Scrutin de liste : la somme des voix des listes doit égaler les exprimés.
    liste = d[d.mode_scrutin == "liste"]
    s = liste.groupby(["tour", "code_insee", "secteur"], dropna=False).agg(voix=("voix", "sum"), exp=("exprimes", "first"))
    out["somme voix listes != exprimes"] = int((s.voix != s.exp).sum())
    return pd.Series(out)


display(pd.DataFrame({y: aberrations_muni(d) for y, d in muni.items()}))

,2014,2020,2026
voix > inscrits,0,0,0
voix > exprimes,0,0,1
voix < 0,0,0,0
"pct_voix_exp hors [0, 100]",0,0,1
votants > inscrits (communes),0,0,1
exprimes > votants (communes),0,0,1
"participation hors [0, 1]",0,0,1
inscrits = 0,0,0,0
inscrits manquants,0,0,0
somme voix listes != exprimes,88,1,31


In [6]:
# Détail des cas « somme des voix != exprimés » (scrutin de liste)
for y, d in muni.items():
    liste = d[d.mode_scrutin == "liste"]
    s = liste.groupby(["tour", "code_insee", "secteur", "libelle_commune"], dropna=False).agg(
        n_listes=("voix", "size"), voix=("voix", "sum"), exprimes=("exprimes", "first"))
    bad = s[s.voix != s.exprimes]
    print(f"{y} : {len(bad)} cas")
    if len(bad):
        display(bad.assign(ecart=bad.voix - bad.exprimes).head(5))

2014 : 88 cas


n_listes  voix  exprimes  ecart
tour code_insee secteur libelle_commune                                              
1    98713      SN01    Arutua Section 01 APATAKI            4   505       275    230
                SN02    Arutua Section 02 ARUTUA            37  4327       481   3846
                SN03    Arutua Section 03 KAUKURA           18  2129       371   1758
     98716      SN01    Fakarava Section 01 FAKARAVA        54  4824       497   4327
                SN02    Fakarava Section 02 KAUEHI          29  2580       441   2139

2020 : 1 cas


,,,,n_listes,voix,exprimes,ecart
tour,code_insee,secteur,libelle_commune,,,,
2,80240,<NA>,Doingt,3,696,697,-1


2026 : 31 cas


n_listes  voix  exprimes  ecart
tour code_insee secteur libelle_commune                                   
1    04022      <NA>    Barrême                   2   305       303      2
     13038      <NA>    Fontvieille               4  2322      2320      2
     22209      <NA>    Beaussais-sur-Mer         2  2370      2369      1
     26242      <NA>    Le Poët-en-Percip         1    14        15     -1
     27681      <NA>    Vernon                    5  8543      8534      9

In [7]:
# 2014, communes < 1000 hab., 1er tour : le fichier ne contient que les ÉLUS.
# Une commune où personne n'est élu au T1 n'y figure pas (elle n'apparaît qu'au T2).
c14 = communes[communes.annee == 2014]
t1 = set(c14[c14.tour == 1].code_insee)
t2_seul = c14[(c14.tour == 2) & ~c14.code_insee.isin(t1)]
print(f"2014 : {t2_seul.code_insee.nunique()} commune(s) présentes au T2 mais pas au T1")
print("Part d'élus parmi les lignes plurinominales T1 2014 :",
      muni[2014].query("tour == 1 and mode_scrutin == 'plurinominal'").elu.mean())

2014 : 229 commune(s) présentes au T2 mais pas au T1
Part d'élus parmi les lignes plurinominales T1 2014 : 1.0


In [8]:
# 2026 : les noms des têtes de liste sont vides au 1er tour dans le fichier source.
display(muni[2026].groupby("tour")[["nom", "prenom", "sexe", "nuance"]].agg(lambda s: s.notna().mean()).round(3))

,nom,prenom,sexe,nuance
tour,,,,
1,0.0,0.0,0.0,0.18
2,1.0,1.0,1.0,0.642


## 2. Maires : RNE (état actuel) et maires sortants 2026

Clé attendue : un maire par commune (`code_insee`).

In [9]:
for name in ("rne_maires", "maires_sortants_2026"):
    d = get(name)
    print(f"=== {name}")
    diagnostic(d, key=["code_insee"])
    age = (pd.Timestamp("2026-03-15") - d.date_de_naissance).dt.days / 365.25
    print("âge au 15/03/2026 :", age.describe().round(1).to_dict())
    print("âge < 18 ou > 100 :", int(((age < 18) | (age > 100)).sum()))
    print("communes :", d.code_insee.nunique())

=== rne_maires
shape : (34826, 15)


,code_insee,code_du_departement,libelle_du_departement,code_de_la_collectivite_a_statut_particulier,libelle_de_la_collectivite_a_statut_particulier,code_de_la_commune,libelle_de_la_commune,nom,prenom,sexe,date_de_naissance,code_de_la_categorie_socio_professionnelle,libelle_de_la_categorie_socio_professionnelle,date_de_debut_du_mandat,date_de_debut_de_la_fonction
type,string,string,string,string,string,string,string,string,string,string,datetime64[us],string,string,datetime64[us],datetime64[us]
pct_manquant,0.0,0.4,0.4,98.57,98.57,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,code_insee,code_du_departement,libelle_du_departement,code_de_la_collectivite_a_statut_particulier,libelle_de_la_collectivite_a_statut_particulier,code_de_la_commune,libelle_de_la_commune,nom,prenom,sexe,date_de_naissance,code_de_la_categorie_socio_professionnelle,libelle_de_la_categorie_socio_professionnelle,date_de_debut_du_mandat,date_de_debut_de_la_fonction
11833,32098,32,Gers,<NA>,<NA>,32098,Cazaux-Savès,BRUMAS-RETAILLEAU,Véronique,F,1965-06-17,34,"Professeur, profession scientifique",2026-03-15,2026-03-20
22211,59584,59,Nord,<NA>,<NA>,59584,Taisnières-Sur-Hon,JACMAIN,Chantal,F,1959-11-02,33,Cadre de la fonction publique,2026-03-15,2026-03-20
34165,91135,91,Essonne,<NA>,<NA>,91135,Champcueil,JACQUET,Sandrine,F,1968-08-18,56,Personnel des services directs aux particuliers,2026-03-15,2026-03-21


doublons sur ['code_insee'] : 0 lignes (0 clés)
âge au 15/03/2026 : {'count': 34826.0, 'mean': 59.6, 'std': 10.9, 'min': 19.6, '25%': 52.3, '50%': 60.8, '75%': 67.7, 'max': 94.2}
âge < 18 ou > 100 : 0
communes : 34826
=== maires_sortants_2026
shape : (34889, 15)


,code_insee,code_du_departement,libelle_du_departement,code_de_la_collectivite_a_statut_particulier,libelle_de_la_collectivite_a_statut_particulier,code_de_la_commune,libelle_de_la_commune,nom,prenom,sexe,date_de_naissance,code_de_la_categorie_socio_professionnelle,libelle_de_la_categorie_socio_professionnelle,date_de_debut_du_mandat,date_de_debut_de_la_fonction
type,string,string,string,string,string,string,string,string,string,string,datetime64[us],string,string,datetime64[us],datetime64[us]
pct_manquant,0.0,0.4,0.4,98.58,98.58,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,code_insee,code_du_departement,libelle_du_departement,code_de_la_collectivite_a_statut_particulier,libelle_de_la_collectivite_a_statut_particulier,code_de_la_commune,libelle_de_la_commune,nom,prenom,sexe,date_de_naissance,code_de_la_categorie_socio_professionnelle,libelle_de_la_categorie_socio_professionnelle,date_de_debut_du_mandat,date_de_debut_de_la_fonction
25395,65223,65,Hautes-Pyrénées,<NA>,<NA>,65223,Horgues,SÉGNERÉ,Jean-Michel,M,1974-07-16,23,Chef d'entreprise de 10 salariés ou plus,2020-05-18,2020-05-28
14679,39375,39,Jura,<NA>,<NA>,39375,Moutonne,PARIS,Robert,M,1958-01-30,77,Ancien employé,2020-06-28,2020-07-03
6802,21306,21,Côte-D'Or,<NA>,<NA>,21306,Grenant-Lès-Sombernon,LECOUR,Jean-Luc,M,1949-06-22,71,Ancien agriculteur exploitant,2020-05-18,2020-05-23


doublons sur ['code_insee'] : 2 lignes (1 clés)
âge au 15/03/2026 : {'count': 34889.0, 'mean': 64.4, 'std': 10.3, 'min': 24.7, '25%': 57.4, '50%': 65.8, '75%': 72.1, 'max': 94.2}
âge < 18 ou > 100 : 0
communes : 34888


## 3. Candidatures et élus

Clés : liste = commune x tour x numéro de panneau ; candidat = liste x ordre.

In [10]:
cand20 = get("candidatures_2020")
diagnostic(cand20, key=["code_insee", "n_panneau_liste", "n_candidat"])
tetes20 = cand20[cand20.tete_de_liste]
print("têtes de liste 2020 :", len(tetes20), "| communes :", tetes20.code_insee.nunique())
print("listes avec != 1 tête :", int((cand20.groupby(["code_insee", "n_panneau_liste"]).tete_de_liste.sum() != 1).sum()))

shape : (546142, 17)


,code_insee,secteur,code_du_departement,libelle_du_departement,code_commune,libelle_commune,n_panneau_liste,libelle_abrege_liste,libelle_etendu_liste,nuance_liste,n_candidat,sexe_candidat,nom_candidat,prenom_candidat,nationalite,candidat_au_conseil_communautaire,tete_de_liste
type,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,boolean
pct_manquant,0.0,98.17,0.0,0.0,0.0,0.0,0.0,0.0,0.04,38.21,0.0,0.0,0.0,0.0,0.01,0.02,0.01


,code_insee,secteur,code_du_departement,libelle_du_departement,code_commune,libelle_commune,n_panneau_liste,libelle_abrege_liste,libelle_etendu_liste,nuance_liste,n_candidat,sexe_candidat,nom_candidat,prenom_candidat,nationalite,candidat_au_conseil_communautaire,tete_de_liste
156381,34172,<NA>,34,Hérault,172,Montpellier,11,LISTE OUVRIERE D'UNITE «,"Liste ouvrière d'unité ""MONTPELLIER 100% SERVI...",LEXG,59,F,DUVERGER,Nathalie,Française,N,False
232920,49301,<NA>,49,Maine-et-Loire,301,Sèvremoine,1,SEVREMOINE ENSEMBLE,SEVREMOINE ENSEMBLE,LDVG,63,F,NAUD,Annie,Française,N,False
17811,06027,<NA>,6,Alpes-Maritimes,27,Cagnes-sur-Mer,4,CAGNES-ÉCOLOGIE,CAGNES-ÉCOLOGIE,LECO,18,F,ROCHE,Jacqueline,Française,N,False


doublons sur ['code_insee', 'n_panneau_liste', 'n_candidat'] : 9800 lignes (1264 clés)
têtes de liste 2020 : 20801 | communes : 9958
listes avec != 1 tête : 36


In [11]:
diagnostic(get("candidats_plurinominal_2020"), key=["code_insee", "n_depot_candidat"])

shape : (356352, 11)


,code_insee,secteur,code_du_departement,libelle_du_departement,code_commune,libelle_commune,n_depot_candidat,sexe_candidat,nom_candidat,prenom_candidat,nationalite
type,string,string,str,str,str,str,str,str,str,str,str
pct_manquant,0.0,99.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,code_insee,secteur,code_du_departement,libelle_du_departement,code_commune,libelle_commune,n_depot_candidat,sexe_candidat,nom_candidat,prenom_candidat,nationalite
15256,03206,<NA>,03,Allier,206,La Petite-Marche,7,F,AUCOUTURIER,Marie-Ange,Française
316341,79032,<NA>,79,Deux-Sèvres,32,Béceleuf,18,M,PINTAUD,Alexis,Française
226952,59125,<NA>,59,Nord,125,Cantaing-sur-Escaut,6,M,CAILLIEZ,Pascal,Française


doublons sur ['code_insee', 'n_depot_candidat'] : 248 lignes (108 clés)


In [12]:
elus20 = get("elus_2020")
diagnostic(elus20, key=["code_insee", "nom", "prenom", "date_de_naissance"])
print("communes avec élus 2020 :", elus20.code_insee.nunique())

shape : (512266, 10)


,code_insee,secteur,code_departement,code_commune,libelle_de_la_commune,sexe,nom,prenom,date_de_naissance,election_au_conseil_communautaire
type,string,string,str,str,str,str,str,str,datetime64[us],str
pct_manquant,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,code_insee,secteur,code_departement,code_commune,libelle_de_la_commune,sexe,nom,prenom,date_de_naissance,election_au_conseil_communautaire
253520,49053,<NA>,49,53,Brossay,F,LAVILLE,Marie-Joseph,1945-11-16,
89672,18242,<NA>,18,242,Sancoins,F,VALENTI,Nadège,1950-01-13,N
294552,56152,<NA>,56,152,Le Palais,F,PORTUGAL,Aude,1985-08-09,N


doublons sur ['code_insee', 'nom', 'prenom', 'date_de_naissance'] : 0 lignes (0 clés)
communes avec élus 2020 : 34974


In [13]:
cand26 = get("candidatures_2026")
diagnostic(cand26, key=["tour", "code_insee", "numero_de_panneau", "ordre"])
t = cand26[cand26.tete_de_liste]
print("têtes de liste 2026 par tour :", t.groupby("tour").size().to_dict())
print("communes avec candidatures par tour :", cand26.groupby("tour").code_insee.nunique().to_dict())
n_tete = cand26.groupby(["tour", "code_insee", "numero_de_panneau"]).tete_de_liste.sum()
print("listes avec != 1 tête :", int((n_tete != 1).sum()))

shape : (1011936, 19)


,code_insee,tour,code_departement,departement,code_circonscription,circonscription,numero_de_panneau,libelle_abrege_de_liste,libelle_de_la_liste,code_nuance_de_liste,nuance_de_liste,tete_de_liste,ordre,sexe,nom_sur_le_bulletin_de_vote,prenom_sur_le_bulletin_de_vote,nationalite,code_personnalite,cc
type,string,int64,str,str,str,str,str,str,str,str,str,boolean,str,str,str,str,str,str,str
pct_manquant,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,code_insee,tour,code_departement,departement,code_circonscription,circonscription,numero_de_panneau,libelle_abrege_de_liste,libelle_de_la_liste,code_nuance_de_liste,nuance_de_liste,tete_de_liste,ordre,sexe,nom_sur_le_bulletin_de_vote,prenom_sur_le_bulletin_de_vote,nationalite,code_personnalite,cc
667808,72364,1,72,Sarthe,72364,Vaas,2,DV,Dynamique Védaquaise,,,False,3,F,HOUR,Céline,Française,,
41947,06083,1,06,Alpes-Maritimes,06083,Menton,6,PRINTEMPS MENTONNAIS,PRINTEMPS MENTONNAIS,LUG,Liste d'union à gauche,True,1,M,LANQUAR-CASTIEL,Laurent,Française,,OUI
1011299,98821,2,988,Nouvelle-Calédonie,98821,Païta,6,LISTE CITOYENNE FLNKS PAITA,LISTE CITOYENNE FLNKS PAITA,LREG,Liste régionaliste,False,37,F,PAITA,Hélène Maéta,Française,,


doublons sur ['tour', 'code_insee', 'numero_de_panneau', 'ordre'] : 297 lignes (67 clés)
têtes de liste 2026 par tour : {1: 49911, 2: 4208}
communes avec candidatures par tour : {1: 34904, 2: 1632}
listes avec != 1 tête : 178


In [14]:
elus26 = get("candidats_elus_2026")
diagnostic(elus26, key=["tour", "code_insee", "nompsn", "prepsn", "date_naissance"])
print("communes avec élus 2026 par tour :", elus26.groupby("tour").code_insee.nunique().to_dict())

shape : (511444, 14)


,code_insee,tour,nompsn,prepsn,date_naissance,sexpsn,codpro,effectif_legal,coddpt,codcom,ind_elu_epci,code_nuance_de_liste,code_personnalite,tour_election
type,string,int64,str,str,datetime64[us],str,str,str,str,str,str,str,str,str
pct_manquant,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,code_insee,tour,nompsn,prepsn,date_naissance,sexpsn,codpro,effectif_legal,coddpt,codcom,ind_elu_epci,code_nuance_de_liste,code_personnalite,tour_election
57495,14240,1,AUBRIS,Isabelle,1974-02-23,F,56,11,14,14240,,,,élu T1
494026,65460,2,DERRE,Xavier,1973-06-14,M,23,27,65,65460,OUI,LDVC,,élu T2
172881,35176,1,PAUMIER,Vanessa,1981-06-17,F,37,33,35,35176,,LDVG,,élu T1


doublons sur ['tour', 'code_insee', 'nompsn', 'prepsn', 'date_naissance'] : 0 lignes (0 clés)
communes avec élus 2026 par tour : {1: 33310, 2: 1526}


## 4. Code officiel géographique 2026

`cog_mouvements` (v_mvt_commune_2026) est chargé mais la table de passage n'est
**pas** reconstruite ici.

In [15]:
cog = get("cog_communes")
diagnostic(cog, key=["typecom", "code_insee"])
display(cog.typecom.value_counts().to_frame())

shape : (37496, 12)


,typecom,code_insee,reg,dep,ctcd,arr,tncc,ncc,nccenr,libelle,can,comparent
type,string,string,string,string,string,string,string,string,string,string,string,string
pct_manquant,0.0,0.0,6.87,6.87,6.87,6.92,0.0,0.0,0.0,0.0,7.02,93.01


,typecom,code_insee,reg,dep,ctcd,arr,tncc,ncc,nccenr,libelle,can,comparent
25846,COM,62455,32,62,62D,626,1,HOCQUINGHEN,Hocquinghen,Hocquinghen,6219,<NA>
21932,COM,55298,44,55,55D,551,0,LOISEY,Loisey,Loisey,5515,<NA>
24945,COM,61072,28,61,61D,613,0,CANAPVILLE,Canapville,Canapville,6121,<NA>


doublons sur ['typecom', 'code_insee'] : 0 lignes (0 clés)


,count
typecom,
COM,34875
COMD,2105
COMA,471
ARM,45


In [16]:
mvt = get("cog_mouvements")
diagnostic(mvt)
# MOD 31-34 : fusions, création de communes nouvelles ; 21 : rétablissement
display(mvt[mvt.date_eff >= "2014-01-01"].groupby([mvt.date_eff.dt.year, "mod"]).size().unstack(fill_value=0))

shape : (13734, 14)


,mod,date_eff,typecom_av,com_av,tncc_av,ncc_av,nccenr_av,libelle_av,typecom_ap,com_ap,tncc_ap,ncc_ap,nccenr_ap,libelle_ap
type,string,datetime64[us],string,string,string,string,string,string,string,string,string,string,string,string
pct_manquant,0.0,0.0,0.92,0.92,0.92,0.92,0.92,0.92,0.01,0.01,0.01,0.01,0.01,0.01


,mod,date_eff,typecom_av,com_av,tncc_av,ncc_av,nccenr_av,libelle_av,typecom_ap,com_ap,tncc_ap,ncc_ap,nccenr_ap,libelle_ap
3232,32,2017-01-01,COMD,50642,0,VINDEFONTAINE,Vindefontaine,Vindefontaine,COMD,50642,0,VINDEFONTAINE,Vindefontaine,Vindefontaine
7611,31,1977-02-15,COM,74246,0,SAINT MARTIN SUR ARVE,Saint-Martin-sur-Arve,Saint-Martin-sur-Arve,COM,74256,0,SALLANCHES,Sallanches,Sallanches
12816,31,1961-01-01,COM,24082,0,CARSAC DE CARLUX,Carsac-de-Carlux,Carsac-de-Carlux,COM,24082,0,CARSAC AILLAC,Carsac-Aillac,Carsac-Aillac


mod,10,21,32,34,35,41,50,70,71,72
date_eff,,,,,,,,,,
2014,8,6,0,26,4,0,2,6,0,0
2015,5,0,224,13,7,1,0,0,0,0
2016,4,0,2217,5,3,1,0,0,0,0
2017,16,3,1211,13,12,0,0,0,0,0
2018,11,0,208,6,2,2,0,0,0,0
2019,0,5,1216,1,22,0,0,0,0,0
2020,11,0,0,13,228,0,0,0,14,0
2021,0,0,10,1,43,0,1,1,0,0
2022,11,0,32,3,27,0,0,0,0,0


In [17]:
hist = get("cog_historique")
diagnostic(hist, key=["typecom", "code_insee", "date_debut"])

shape : (42333, 8)


,typecom,code_insee,tncc,ncc,nccenr,libelle,date_debut,date_fin
type,string,string,string,string,string,string,datetime64[us],datetime64[us]
pct_manquant,0.0,0.0,0.0,0.0,0.0,0.0,0.0,82.49


,typecom,code_insee,tncc,ncc,nccenr,libelle,date_debut,date_fin
27623,COM,60310,1,HERCHIES,Herchies,Herchies,1943-01-01,NaT
37760,COM,79331,0,TOURTENAY,Tourtenay,Tourtenay,1943-01-01,NaT
33416,COM,71015,1,AUXY,Auxy,Auxy,1943-01-01,NaT


doublons sur ['typecom', 'code_insee', 'date_debut'] : 0 lignes (0 clés)


## 5. Contours des communes (2025, 50 m)

In [18]:
import geopandas as gpd

contours = gpd.read_parquet(PROCESSED / "contours_communes.parquet")
diagnostic(contours.drop(columns="geometry"), key=["code_insee"])
cog_com = set(cog.loc[cog.typecom == "COM", "code_insee"])
print("communes COG 2026 sans contour 2025 :", len(cog_com - set(contours.code_insee)))
print("contours sans commune COG 2026 :", len(set(contours.code_insee) - cog_com))
print("géométries invalides :", int((~contours.is_valid).sum()), "| vides :", int(contours.is_empty.sum()))

shape : (35014, 7)


,code_insee,nom,departement,region,commune,plm,epci
type,string,str,str,str,str,float64,str
pct_manquant,0.0,0.0,0.0,0.0,99.87,99.99,0.41


,code_insee,nom,departement,region,commune,plm,epci
32345,83102,Régusse,83,93,NaN,NaN,200040210
24637,63287,Les Pradeaux,63,84,NaN,NaN,200070407
16123,43208,Sainte-Marguerite,43,84,NaN,NaN,200073393


doublons sur ['code_insee'] : 0 lignes (0 clés)
communes COG 2026 sans contour 2025 : 0
contours sans commune COG 2026 : 139
géométries invalides : 0 | vides : 0


## 6. INSEE : Filosofi, diplômes, population

Fichiers Melodi déjà filtrés sur le niveau géographique « commune » (`GEO_OBJECT == "COM"`)
dans `src/load.py`.

In [19]:
filo = get("filosofi")
diagnostic(filo, key=["code_insee", "annee"])
print(filo[["med_sl", "pr_md60"]].describe().round(1))
print("taux de pauvreté hors [0, 100] :", int((~filo.pr_md60.dropna().between(0, 100)).sum()))

shape : (34768, 4)


,code_insee,annee,med_sl,pr_md60
type,string,Int64,float64,float64
pct_manquant,0.0,0.0,11.43,84.92


,code_insee,annee,med_sl,pr_md60
24154,62721,2023,26280.0,NaN
5819,17296,2023,24800.0,NaN
15824,42256,2023,26810.0,9.0


doublons sur ['code_insee', 'annee'] : 0 lignes (0 clés)
        med_sl  pr_md60
count  30794.0   5242.0
mean   26257.4     13.1
std     3526.9      7.1
min    12420.0      3.0
25%    24090.0      8.0
50%    25780.0     11.0
75%    27830.0     17.0
max    62730.0     53.0
taux de pauvreté hors [0, 100] : 0


In [20]:
dipl = get("diplomes")
diagnostic(dipl, key=["code_insee", "annee", "sex", "educ"])
print("communes par année :", dipl.groupby("annee").code_insee.nunique().to_dict())
print("valeurs négatives :", int((dipl.valeur < 0).sum()))

shape : (2718924, 9)


,code_insee,age,sex,educ,rp_measure,freq,obs_status,annee,valeur
type,string,category,category,category,category,category,category,Int64,float64
pct_manquant,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.01


,code_insee,age,sex,educ,rp_measure,freq,obs_status,annee,valeur
637507,24372,Y_GE15,M,500T702_RP,POP,A,A,2012,310.31659
344028,14014,Y_GE15,F,_T,POP,A,A,2017,392.31956
1699752,59048,Y_GE15,F,200_RP,POP,A,A,2023,21.88274


doublons sur ['code_insee', 'annee', 'sex', 'educ'] : 0 lignes (0 clés)
communes par année : {np.int64(2012): 34858, np.int64(2017): 34858, np.int64(2023): 34858}
valeurs négatives : 0


In [21]:
pop = get("population")
diagnostic(pop, key=["code_insee", "annee", "sex", "pcs", "age"])
print("communes par année :", pop.groupby("annee").code_insee.nunique().to_dict())
print("valeurs négatives :", int((pop.valeur < 0).sum()))

shape : (11293992, 9)


,code_insee,sex,pcs,age,rp_measure,freq,obs_status,annee,valeur
type,string,category,category,category,category,category,category,Int64,float64
pct_manquant,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.01


,code_insee,sex,pcs,age,rp_measure,freq,obs_status,annee,valeur
1510144,14362,M,4,Y_GE55,POP,A,A,2023,0.0
8555478,67394,F,7,Y25T54,POP,A,A,2023,0.0
7192482,59471,F,3,Y15T24,POP,A,A,2012,0.0


doublons sur ['code_insee', 'annee', 'sex', 'pcs', 'age'] : 0 lignes (0 clés)
communes par année : {np.int64(2012): 34858, np.int64(2017): 34858, np.int64(2023): 34858}
valeurs négatives : 0


## 7. Présidentielles 2017 et 2022 (une ligne par commune x candidat)

In [22]:
for y in (2017, 2022):
    d = get(f"presidentielle_{y}")
    print(f"=== {y}")
    diagnostic(d, key=["code_insee", "annee", "tour", "num_panneau"])
    c = d.drop_duplicates(["code_insee", "tour"])
    s = d.groupby(["code_insee", "tour"]).agg(voix=("voix", "sum"), exp=("exprimes", "first"))
    print(pd.Series({
        "communes T1 / T2": f"{c[c.tour == 1].shape[0]} / {c[c.tour == 2].shape[0]}",
        "voix > inscrits": int((d.voix > d.inscrits).sum()),
        "votants > inscrits": int((c.votants > c.inscrits).sum()),
        "participation hors [0, 1]": int((~(c.votants / c.inscrits).dropna().between(0, 1)).sum()),
        "somme voix != exprimes": int((s.voix != s.exp).sum()),
    }).to_string())

=== 2017
shape : (461448, 20)


,code_insee,code_dep,libelle_dep,code_commune_source,libelle_commune,inscrits,abstentions,votants,blancs,nuls,exprimes,num_panneau,sexe,nom,prenom,voix,pct_voix_ins,pct_voix_exp,annee,tour
type,string,string,str,string,str,Int64,Int64,Int64,Int64,Int64,Int64,Int64,str,str,str,Int64,float64,float64,int64,int64
pct_manquant,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,code_insee,code_dep,libelle_dep,code_commune_source,libelle_commune,inscrits,abstentions,votants,blancs,nuls,exprimes,num_panneau,sexe,nom,prenom,voix,pct_voix_ins,pct_voix_exp,annee,tour
426214,49253,49,Maine-et-Loire,253,Le Puy-Notre-Dame,843,201,642,57,21,564,1,M,MACRON,Emmanuel,328,38.91,58.16,2017,2
347289,80176,80,Somme,176,Carrépuis,208,34,174,3,1,170,10,M,ASSELINEAU,François,1,0.48,0.59,2017,1
359826,83017,83,Var,17,Belgentier,1887,387,1500,26,8,1466,4,M,HAMON,Benoît,52,2.76,3.55,2017,1


doublons sur ['code_insee', 'annee', 'tour', 'num_panneau'] : 0 lignes (0 clés)
communes T1 / T2             35496 / 35496
voix > inscrits                          0
votants > inscrits                       0
participation hors [0, 1]                0
somme voix != exprimes                   0
=== 2022
shape : (490490, 21)


,code_insee,code_dep,libelle_dep,code_commune_source,libelle_commune,etat_saisie,inscrits,abstentions,votants,blancs,nuls,exprimes,num_panneau,sexe,nom,prenom,voix,pct_voix_ins,pct_voix_exp,annee,tour
type,string,string,str,string,str,str,Int64,Int64,Int64,Int64,Int64,Int64,Int64,str,str,str,Int64,float64,float64,int64,int64
pct_manquant,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,code_insee,code_dep,libelle_dep,code_commune_source,libelle_commune,etat_saisie,inscrits,abstentions,votants,blancs,nuls,exprimes,num_panneau,sexe,nom,prenom,voix,pct_voix_ins,pct_voix_exp,annee,tour
487688,88364,88,Vosges,364,Puzieux,Complet,136,16,120,12,2,106,1,M,MACRON,Emmanuel,43,31.62,40.57,2022,2
371605,79337,79,Deux-Sèvres,337,Le Vanneau-Irleau,Complet,745,162,583,14,1,568,2,M,ROUSSEL,Fabien,17,2.28,2.99,2022,1
327533,70053,70,Haute-Saône,053,Les Bâties,Complet,79,17,62,0,1,61,6,M,ZEMMOUR,Éric,2,2.53,3.28,2022,1


doublons sur ['code_insee', 'annee', 'tour', 'num_panneau'] : 0 lignes (0 clés)
communes T1 / T2             35035 / 35035
voix > inscrits                          0
votants > inscrits                       0
participation hors [0, 1]                0
somme voix != exprimes                   0


## 8. Squelette du panel

Une ligne par (commune, scrutin municipal) : tour décisif, inscrits et votants
par tour. Les secteurs (PLM) et sections (Polynésie) sont sommés au niveau de la
commune. Aucune variable explicative à ce stade.

In [23]:
par_tour = (communes.groupby(["code_insee", "annee", "tour"], as_index=False)
            .agg(inscrits=("inscrits", "sum"), votants=("votants", "sum"),
                 libelle_commune=("libelle_commune", "first")))
panel = par_tour.pivot_table(index=["code_insee", "annee"], columns="tour",
                             values=["inscrits", "votants"], aggfunc="first")
panel.columns = [f"{v}_t{t}" for v, t in panel.columns]
panel = panel.reset_index()
panel["tour_final"] = np.where(panel.inscrits_t2.notna(), 2, 1)
panel = panel.merge(par_tour.groupby(["code_insee", "annee"]).libelle_commune.first().reset_index(),
                    on=["code_insee", "annee"])
for c in ("inscrits_t1", "votants_t1", "inscrits_t2", "votants_t2"):
    panel[c] = panel[c].astype("Int64")
assert not panel.duplicated(["code_insee", "annee"]).any()
print(panel.shape)
display(panel.head())

(106591, 8)


,code_insee,annee,inscrits_t1,inscrits_t2,votants_t1,votants_t2,tour_final,libelle_commune
0,01001,2014,599,<NA>,355,<NA>,1,L'Abergement-Clémenciat
1,01001,2020,622,<NA>,282,<NA>,1,L'Abergement-Clémenciat
2,01001,2026,679,<NA>,374,<NA>,1,L'Abergement-Clémenciat
3,01002,2014,226,<NA>,189,<NA>,1,L'Abergement-de-Varey
4,01002,2020,212,<NA>,129,<NA>,1,L'Abergement-de-Varey


In [24]:
print("communes par scrutin :")
display(panel.groupby("annee").agg(n_communes=("code_insee", "nunique"),
                                   dont_2nd_tour=("tour_final", lambda s: int((s == 2).sum()))))

communes par scrutin :


,n_communes,dont_2nd_tour
annee,,
2014,36749,6508
2020,35006,4579
2026,34836,1526


In [25]:
# Présence par scrutin : combinaisons (2014, 2020, 2026)
presence = panel.assign(v=True).pivot_table(index="code_insee", columns="annee", values="v",
                                            aggfunc="first", fill_value=False).astype(bool)
combo = presence.apply(lambda r: "".join("X" if r[y] else "." for y in (2014, 2020, 2026)), axis=1)
display(combo.value_counts().rename_axis("2014 2020 2026").to_frame("n_communes"))

,n_communes
2014 2020 2026,
XXX,34800
X..,1728
XX.,195
X.X,26
.XX,10
.X.,1


In [26]:
# Communes absentes d'au moins un scrutin, avec indice de fusion issu du COG
# (consultation de v_mvt_commune_2026 uniquement ; table de passage non reconstruite).
FUSION_MODS = {"31", "32", "33", "34"}
mvt_fusion = mvt[mvt["mod"].isin(FUSION_MODS) & (mvt.date_eff >= "2014-01-01")]
absorbee = mvt_fusion[mvt_fusion.com_av != mvt_fusion.com_ap].groupby("com_av").agg(
    devenue=("com_ap", "first"), date_fusion=("date_eff", "first"))
actuelles = set(cog.loc[cog.typecom == "COM", "code_insee"])

incomplet = combo[combo != "XXX"].rename("presence").to_frame()
incomplet = incomplet.join(absorbee, how="left")
incomplet["commune_cog_2026"] = incomplet.index.isin(actuelles)
incomplet["outre_mer_hors_cog"] = incomplet.index.str[:3].isin(["975", "977", "978", "986", "987", "988"])
incomplet = incomplet.join(panel.groupby("code_insee").libelle_commune.first())
print(f"{len(incomplet)} communes absentes d'au moins un scrutin")
display(pd.crosstab(incomplet.presence, [incomplet.devenue.notna().rename("fusionnée")]))
display(incomplet.sort_values("presence").head(15))

1960 communes absentes d'au moins un scrutin


fusionnée,False,True
presence,,
.X.,1,0
.XX,10,0
X..,26,1702
X.X,15,11
XX.,97,98


,presence,devenue,date_fusion,commune_cog_2026,outre_mer_hors_cog,libelle_commune
code_insee,,,,,,
98730,.X.,<NA>,NaT,False,True,Napuka Section 01 NAPUKA
33187,.XX,<NA>,NaT,True,False,Gironde-sur-Dropt
42055,.XX,<NA>,NaT,True,False,Châtelus
46210,.XX,<NA>,NaT,True,False,Nadillac
52228,.XX,<NA>,NaT,True,False,Grandchamp
63074,.XX,<NA>,NaT,True,False,Chalus
76601,.XX,<NA>,NaT,True,False,Saint-Lucien
98801,.XX,<NA>,NaT,False,True,Belep
98806,.XX,<NA>,NaT,False,True,Farino


In [27]:
# Synthèse des motifs probables d'absence
motif = np.select(
    [incomplet.devenue.notna(), incomplet.outre_mer_hors_cog, ~incomplet.commune_cog_2026],
    ["fusion / commune nouvelle depuis 2014", "outre-mer hors COG (PF, NC, SPM...)", "absente du COG 2026"],
    default="présente au COG 2026 : donnée manquante dans un scrutin")
display(pd.crosstab(incomplet.presence, motif))

col_0,absente du COG 2026,fusion / commune nouvelle depuis 2014,"outre-mer hors COG (PF, NC, SPM...)",présente au COG 2026 : donnée manquante dans un scrutin
presence,,,,
.X.,0,0,1,0
.XX,0,0,4,6
X..,4,1702,22,0
X.X,0,11,1,14
XX.,4,98,25,68
